# PDF Ingestion
- Ingest PDF reports and extract clean text using pdfplumber
- Made because the initial training data on txt files made from CSV was not semantic enough, it was too structured and we need more unstructured data for semantic analysis

## Setup

In [14]:
# !pip install -r ../requirements.txt

In [15]:
# Imports
import pdfplumber
from pathlib import Path
import re

In [ ]:
# Paths
PROJECT_ROOT    = Path().resolve().parent
PDF_DIR         = PROJECT_ROOT / "data" / "1_source" / "PDF Reports"
CLEANED_DIR     = PROJECT_ROOT / "data" / "2_cleaned" / "PDF_Reports"
KB_REPORTS_DIR  = PROJECT_ROOT / "data" / "3_txt_KB" / "PDF_Reports"
CLEANED_DIR.mkdir(parents=True, exist_ok=True)

pdf_files = [
    "FRA_2025.pdf",
    "ILGA_2023.pdf",
    "ILGA_2024.pdf",
    "ILGA_2025.pdf"
]

## Processing Files
### Helper functions

In [17]:
def clean_text(text: str) -> str:
    # Remove excessive whitespace and line breaks
    text = re.sub(r'\n+', '\n', text)
    text = re.sub(r'[ \t]+', ' ', text)

    # Remove page numbers (simple heuristic)
    text = re.sub(r'\n\d+\n', '\n', text)

    # Remove hyphenation across lines
    text = re.sub(r'-\n', '', text)

    return text.strip()

In [18]:
def extract_text_from_pdf(pdf_path: Path) -> str:
    full_text = []

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            # Extract only text (ignores images/graphs automatically)
            text = page.extract_text()
            if text:
                full_text.append(text)

    return "\n".join(full_text)

## Apply cleaning
- convert PDF to text
- output is in `data/2_cleaned/`

In [ ]:
# for pdf_name in pdf_files:
#     pdf_path = PDF_DIR / pdf_name

#     print(f"Processing: {pdf_name}")

#     raw_text = extract_text_from_pdf(pdf_path)
#     cleaned_text = clean_text(raw_text)

#     output_file = CLEANED_DIR / f"{pdf_name.replace('.pdf', '.txt')}"

#     with open(output_file, "w", encoding="utf-8") as f:
#         f.write(cleaned_text)

#     print(f"Saved to: {output_file}\n")

Processing: FRA_2025.pdf
Saved to: C:\Users\RAZER\Desktop\portfolio-projects\1. RAG\data\3_txt_KB\PDF_Reports\FRA_2025.txt

Processing: FRA_2024.pdf
Saved to: C:\Users\RAZER\Desktop\portfolio-projects\1. RAG\data\3_txt_KB\PDF_Reports\FRA_2024.txt

Processing: FRA_2023.pdf
Saved to: C:\Users\RAZER\Desktop\portfolio-projects\1. RAG\data\3_txt_KB\PDF_Reports\FRA_2023.txt

Processing: ILGA_2025.pdf
Saved to: C:\Users\RAZER\Desktop\portfolio-projects\1. RAG\data\3_txt_KB\PDF_Reports\ILGA_2025.txt



## Note
- Commented out execution as the outputted files have been reviewed and processed, so don't want to overwrite them

# Chunk txt files

In [ ]:
files = [
    "FRA_2025.txt",
    "ILGA_2023.txt",
    "ILGA_2024.txt",
    "ILGA_2025.txt"
]

In [ ]:
# Helper function
def write_section(file_prefix, section_name, content, metadata):
    safe_name = re.sub(r'[^A-Za-z0-9]+', '_', section_name).strip('_')
    filename = f"{file_prefix}__{safe_name}.txt"
    output_path = KB_REPORTS_DIR / filename

    with open(output_path, "w", encoding="utf-8") as f:
        for k, v in metadata.items():
            f.write(f"{k}: {v}\n")
        f.write("\n")
        f.write(content.strip())

In [ ]:
# ---------------------------------------------------------------------------
# ILGA SPLITTING
# ---------------------------------------------------------------------------

COUNTRIES = [
    "ALBANIA","ANDORRA","ARMENIA","AUSTRIA","AZERBAIJAN","BELARUS","BELGIUM",
    "BOSNIA AND HERZEGOVINA","BULGARIA","CROATIA","CYPRUS","CZECHIA","DENMARK",
    "ESTONIA","FINLAND","FRANCE","GEORGIA","GERMANY","GREECE","HUNGARY",
    "ICELAND","IRELAND","ITALY","KAZAKHSTAN","KOSOVO","KYRGYZSTAN","LATVIA",
    "LIECHTENSTEIN","LITHUANIA","LUXEMBOURG","MALTA","MOLDOVA","MONACO",
    "MONTENEGRO","NETHERLANDS","NORTH MACEDONIA","NORWAY","POLAND","PORTUGAL",
    "ROMANIA","RUSSIA","SAN MARINO","SERBIA","SLOVAKIA","SLOVENIA","SPAIN",
    "SWEDEN","SWITZERLAND","TAJIKISTAN","TURKEY","TURKMENISTAN","UKRAINE",
    "UNITED KINGDOM","UZBEKISTAN"
]

INSTITUTIONS = [
    "EUROPEAN UNION",
    "COUNCIL OF EUROPE",
    "UNITED NATIONS",
    "ORGANISATION FOR SECURITY AND COOPERATION IN EUROPE"
]